In [2]:
import math
import time
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import pulp

# ================== Device ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================== Physical constants ==================
G = 6.67430e-11
M_earth = 5.972e24
R_earth = 6.371e6

# Basis vectors on CPU (we'll move them to device when needed)
_K_CPU = torch.tensor([0.0, 0.0, 1.0], dtype=torch.float32)
_I_CPU = torch.tensor([1.0, 0.0, 0.0], dtype=torch.float32)
_J_CPU = torch.tensor([0.0, 1.0, 0.0], dtype=torch.float32)


# ================== Utilities ==================
def format_hms(s: float) -> str:
    s = int(s)
    h = s // 3600
    m = (s % 3600) // 60
    sec = s % 60
    return f"{h}h {m:02d}m {sec:02d}s"


def rotate_vector_around_axis_by_angle_torch(vector: torch.Tensor,
                                             axis: torch.Tensor,
                                             angle_rad: float) -> torch.Tensor:
    """
    Rodrigues rotation formula using torch. Returns a unit-normalized rotated vector.
    vector, axis: shape (3,)
    angle_rad: float
    """
    v = vector
    a = axis / (axis.norm() + 1e-12)
    t = torch.tensor(angle_rad, dtype=v.dtype, device=v.device)
    c = torch.cos(t)
    s = torch.sin(t)
    rotated = v * c + torch.cross(a, v) * s + a * torch.dot(a, v) * (1.0 - c)
    return rotated / (rotated.norm() + 1e-12)


# ================== Trajectory parameter creators (torch) ==================
def create_circular_trajectory_parameters_for_satelites_torch(
    theta_inclination_rad: float,
    raan_rad: float,
    radius_orbit: float,
    device: torch.device,
):
    """
    Satellite circular orbit parameters (torch version).
    Returns:
      center (3,), radius (scalar float), u (3,), v (3,), omega (scalar float)
    """
    I = _I_CPU.to(device)
    J = _J_CPU.to(device)
    K = _K_CPU.to(device)

    center = torch.zeros(3, dtype=torch.float32, device=device)

    # node direction in equatorial plane
    u = I * math.cos(raan_rad) + J * math.sin(raan_rad)
    # vector perpendicular to u in equatorial plane
    f = torch.cross(K, u)
    # rotate into inclined plane
    v = rotate_vector_around_axis_by_angle_torch(f, u, theta_inclination_rad)

    # angular velocity
    omega = 2.0 * math.pi * math.sqrt(radius_orbit**-3 * (G * M_earth))
    return center, float(radius_orbit), u, v, float(omega)


def prepare_rectennas_torch(
    lat_long_array_deg: np.ndarray,
    radius_earth: float = R_earth,
    device: torch.device = device,
):
    """
    Precompute rectenna geometry on torch device.
    lat_long_array_deg: (N_rect,2) numpy array [lat_deg, lon_deg]
    Returns:
      centers (N_rect,3)
      radii   (N_rect,)
      u_rect  (N_rect,3)
      v_rect  (N_rect,3)
    """
    lat_long = torch.from_numpy(lat_long_array_deg.astype(np.float32)).to(device)
    lat_deg = lat_long[:, 0]
    lon_deg = lat_long[:, 1]

    # degrees -> radians
    lat_rad = lat_deg * (math.pi / 180.0)
    lon_rad = lon_deg * (math.pi / 180.0)

    K = _K_CPU.to(device)

    z = radius_earth * torch.sin(lat_rad)         # (N_rect,)
    centers = torch.stack([
        torch.zeros_like(z),
        torch.zeros_like(z),
        z
    ], dim=1)                                     # (N_rect,3)

    radii = radius_earth * torch.cos(lat_rad)     # (N_rect,)

    u_rect = torch.stack([
        torch.cos(lon_rad),
        torch.sin(lon_rad),
        torch.zeros_like(lon_rad)
    ], dim=1)                                     # (N_rect,3)

    v_rect = torch.cross(K.expand_as(u_rect), u_rect)  # (N_rect,3)

    return centers, radii, u_rect, v_rect


# ================== 3D circular trajectory (torch) ==================
def find_position_in_space_with_3d_circular_trajectory_torch(
    center: torch.Tensor,
    radius: float,
    u: torch.Tensor,
    v: torch.Tensor,
    omega: float,
    time: torch.Tensor,
    phase: float = 0.0,
):
    """
    torch version.
    time: 1D tensor (K,)
    Returns:
      positions: (K,3)
    """
    t = time
    ang = omega * t + phase
    c = torch.cos(ang)
    s = torch.sin(ang)
    # (K,3)
    return center + radius * (c[:, None] * u[None, :] + s[:, None] * v[None, :])


# ================== Fast ray similarity (torch) ==================
def rays_are_similar_fast_torch(arr_u: torch.Tensor,
                                arr_v: torch.Tensor,
                                cos_theta_threshold: float) -> torch.Tensor:
    """
    Torch vectorized comparison of rays.
    arr_u, arr_v: shape (...,3)
    Returns boolean mask where:
      dot(u,v) > 0 and angle(u,v) < threshold
    via cosine threshold.
    """
    dot = (arr_u * arr_v).sum(dim=-1)           # (...)
    nu = arr_u.norm(dim=-1)                     # (...)
    nv = arr_v.norm(dim=-1)                     # (...)
    denom = nu * nv
    valid = denom > 0.0

    cos_vals = torch.empty_like(dot)
    cos_vals[~valid] = -1.0
    cos_vals[valid] = dot[valid] / denom[valid]

    return (dot > 0.0) & (cos_vals > cos_theta_threshold)


# ================== Single (theta, raan) simulation (torch) ==================
def simulate_one_theta_raan_torch(
    theta_inclination_deg: float,
    raan_deg: float,
    total_time_seconds: int,
    lat_long_array_deg: np.ndarray,
    theta_threshold_deg: float = 10.0,
    chunk_size: int = 3600,
    orbit_height_m: float = 500e3,
    device: torch.device = device,
) -> torch.Tensor:
    """
    Run full simulation for ONE satellite (theta, raan) on GPU/CPU using torch.

    lat_long_array_deg: shape (N_rect,2) in degrees.
    Returns:
      tensor of shape (N_rect, 2):
        column 0: counts (float)
        column 1: ratios (float)
    """
    N_rect = lat_long_array_deg.shape[0]

    # angles in radians
    theta_inclination_rad = theta_inclination_deg * (math.pi / 180.0)
    raan_rad = raan_deg * (math.pi / 180.0)
    theta_threshold_rad = theta_threshold_deg * (math.pi / 180.0)
    cos_theta_threshold = math.cos(theta_threshold_rad)

    # satellite orbit radius
    radius_orbit = R_earth + orbit_height_m

    # satellite parameters
    sat_center, sat_radius, sat_u, sat_v, sat_omega = \
        create_circular_trajectory_parameters_for_satelites_torch(
            theta_inclination_rad=theta_inclination_rad,
            raan_rad=raan_rad,
            radius_orbit=radius_orbit,
            device=device,
        )

    # rectennas precomputation (torch)
    centers, radii, u_rect, v_rect = prepare_rectennas_torch(lat_long_array_deg, device=device)

    # times on device
    times = torch.arange(total_time_seconds, dtype=torch.float32, device=device)

    # counts per rectenna
    counts = torch.zeros(N_rect, dtype=torch.float32, device=device)

    # earth rotation
    omega_rect = 2.0 * math.pi / 86400.0
    phase_rect = 0.0

    # loop over time in chunks to control memory
    for t0 in range(0, total_time_seconds, chunk_size):
        t1 = min(total_time_seconds, t0 + chunk_size)
        t_chunk = times[t0:t1]                         # (K,)
        K = t_chunk.shape[0]

        # satellite positions: (K,3)
        pos_sats_chunk = find_position_in_space_with_3d_circular_trajectory_torch(
            sat_center, sat_radius, sat_u, sat_v, sat_omega,
            time=t_chunk, phase=0.0
        )  # (K,3)

        # rectenna positions for this chunk: fully vectorized
        ang_r = omega_rect * t_chunk + phase_rect      # (K,)
        cr = torch.cos(ang_r)                          # (K,)
        sr = torch.sin(ang_r)                          # (K,)

        # term(t, i, :) = u_rect(i,:)*cos(t) + v_rect(i,:)*sin(t)
        term = (
            cr[:, None, None] * u_rect[None, :, :] +
            sr[:, None, None] * v_rect[None, :, :]
        )  # (K, N_rect, 3)

        pos_rect_chunk = centers[None, :, :] + radii[None, :, None] * term  # (K, N_rect, 3)

        # broadcast satellite positions to (K, N_rect, 3)
        pos_sats_broadcast = pos_sats_chunk[:, None, :]                     # (K,1,3)

        # similarity mask: (K, N_rect)
        mask = rays_are_similar_fast_torch(pos_rect_chunk, pos_sats_broadcast, cos_theta_threshold)

        # accumulate counts per rectenna
        counts += mask.sum(dim=0).float()

    ratios = counts / float(total_time_seconds)  # (N_rect,)
    # (N_rect,2): first column = counts, second = ratios
    out = torch.stack([counts, ratios], dim=1)
    return out


# ================== Grid over theta_list x raan_list (torch, tensor output) ==================
def compute_satellite_grid_torch(
    theta_list_deg: np.ndarray,
    raan_list_deg: np.ndarray,
    total_time_seconds: int,
    lat_long_array_deg: np.ndarray,
    theta_threshold_deg: float = 10.0,
    chunk_size: int = 3600,
    orbit_height_m: float = 500e3,
):
    """
    Build an N_theta x N_raan grid over theta_list and raan_list with torch on GPU/CPU.

    theta_list_deg: array-like, shape (N_theta,)
    raan_list_deg:  array-like, shape (N_raan,)
    lat_long_array_deg: shape (N_rect,2)

    Returns:
      result_tensor: torch.Tensor of shape (N_theta, N_raan, N_rect, 2)
        For each [i,j], result_tensor[i,j] is (N_rect,2):
          [:,0] -> counts
          [:,1] -> ratios
    """
    theta_list_deg = np.asarray(theta_list_deg, dtype=float)
    raan_list_deg  = np.asarray(raan_list_deg, dtype=float)

    N_theta = theta_list_deg.shape[0]
    N_raan  = raan_list_deg.shape[0]
    N_rect  = lat_long_array_deg.shape[0]

    # Preallocate final tensor on CPU (to not blow up GPU memory)
    result_tensor = torch.empty(
        (N_theta, N_raan, N_rect, 2), dtype=torch.float32, device="cpu"
    )

    total_pairs = N_theta * N_raan
    start_time = time.time()

    with tqdm(total=total_pairs, desc="Theta-RAAN grid") as pbar:
        done = 0
        for i in range(N_theta):
            for j in range(N_raan):
                theta_val = theta_list_deg[i]
                raan_val  = raan_list_deg[j]

                # run simulation on GPU/CPU
                out_ij = simulate_one_theta_raan_torch(
                    theta_val,
                    raan_val,
                    total_time_seconds,
                    lat_long_array_deg,
                    theta_threshold_deg=theta_threshold_deg,
                    chunk_size=chunk_size,
                    orbit_height_m=orbit_height_m,
                    device=device,
                )

                # move to CPU and store
                result_tensor[i, j] = out_ij.to("cpu")

                # update progress & ETA
                done += 1
                pbar.update(1)
                elapsed = time.time() - start_time
                rate = elapsed / max(done, 1)
                eta  = rate * (total_pairs - done)
                pbar.set_postfix({
                    "elapsed": format_hms(elapsed),
                    "eta": format_hms(eta)
                })

    print("Total elapsed:", format_hms(time.time() - start_time))
    return result_tensor


# ================== MILP: minimum selection ==================
def solve_minimum_selection(
    data,
    val_threshold,
    verbose=False,
    gap_rel=0.001,
    time_limit=None,
):
    """
    Solve the minimal selection problem.

    Parameters
    ----------
    data : np.ndarray or torch.Tensor
        Tensor of shape (N_theta, N_raan, N_rect, 2).
        The last dimension has 2 "columns"; we ignore column 0 and use column 1.
        After ignoring we conceptually have (N_theta, N_raan, N_rect, 1),
        i.e. for each (theta_idx, raan_idx, rect_idx) we have a scalar in [0,1].
    val_threshold : float or np.ndarray
        Either a scalar threshold or an array of shape (N_rect,)
        specifying the minimum required sum for each rectenna.
    verbose : bool
        If True, prints MILP solver log.
    gap_rel : float
        Relative MIP gap at which CBC is allowed to stop, e.g. 0.001 = 0.1%.
    time_limit : float or None
        Optional time limit in seconds for CBC.

    Returns
    -------
    selected_indices : list of (theta_idx, raan_idx)
        List of grid points (theta, raan) that are selected in the (near-)optimal solution.
    status : str
        Solver status string from pulp.
    """

    # convert data to numpy
    if isinstance(data, torch.Tensor):
        vals_full = data.cpu().numpy()
    else:
        vals_full = np.asarray(data, dtype=float)

    # 1. Extract 2nd column → shape (N_theta, N_raan, N_rect, 1)
    vals = vals_full[..., 1:2]  # keep as last-dim=1 for clarity

    N_theta, N_raan, N_rect, _ = vals.shape
    N_samples = N_theta * N_raan

    # 2. Flatten over (theta, raan) → shape (N_samples, N_rect)
    vals_flat = vals.reshape(N_samples, N_rect)

    # 3. Handle threshold
    if np.isscalar(val_threshold):
        threshold = np.full(N_rect, float(val_threshold))
    else:
        threshold = np.asarray(val_threshold, dtype=float)
        if threshold.shape != (N_rect,):
            raise ValueError(
                f"val_threshold must be scalar or shape (N_rect,), got {threshold.shape}"
            )

    # 4. Define ILP: minimize sum x_s, subject to:
    #    sum_s x_s * vals_flat[s, r] >= threshold[r] for all rectenna r
    prob = pulp.LpProblem("MinimumSelection", pulp.LpMinimize)

    # Binary decision variables: x_s ∈ {0,1}
    x = pulp.LpVariable.dicts(
        "x", range(N_samples), lowBound=0, upBound=1, cat=pulp.LpBinary
    )

    # Objective: minimize total number of selected samples
    prob += pulp.lpSum(x[s] for s in range(N_samples)), "Minimize_number_of_samples"

    # Constraints: for each rectenna r, sum_s x_s * vals_flat[s,r] >= threshold[r]
    for r in range(N_rect):
        prob += (
            pulp.lpSum(vals_flat[s, r] * x[s] for s in range(N_samples))
            >= threshold[r],
            f"rectenna_{r}_coverage",
        )

    # 5. Solve with CBC, enforcing relative gap
    solver = pulp.PULP_CBC_CMD(
        msg=verbose,
        gapRel=gap_rel,          # <-- 0.001 = 0.1% relative MIP gap
        timeLimit=time_limit,    # optional, can be None
    )
    prob.solve(solver)

    status = pulp.LpStatus[prob.status]
    print("MILP status:", status)

    # 6. Extract chosen samples (best incumbent, even if not proven optimal)
    chosen_samples = [s for s in range(N_samples) if pulp.value(x[s]) > 0.5]

    selected_indices = []
    for s in chosen_samples:
        theta_idx = s // N_raan
        raan_idx = s % N_raan
        selected_indices.append((theta_idx, raan_idx))

    return selected_indices, status


# ================== Verification of solution ==================
def verify_solution(
    data,
    selected_indices,
    val_threshold,
    atol=1e-6,
    print_details=True,
):
    """
    Verify that the selected (theta, raan) pairs satisfy the coverage constraints.

    Parameters
    ----------
    data : np.ndarray or torch.Tensor
        Shape (N_theta, N_raan, N_rect, 2). We use column 1 (ratios).
    selected_indices : list of (theta_idx, raan_idx)
        Output of solve_minimum_selection.
    val_threshold : float or np.ndarray
        Same threshold used in the MILP.
    atol : float
        Numerical tolerance when checking >= threshold - atol.
    print_details : bool
        If True, prints per-rectenna coverage and pass/fail.

    Returns
    -------
    coverage : np.ndarray of shape (N_rect,)
        The total summed ratio for each rectenna across the selected points.
    threshold : np.ndarray of shape (N_rect,)
        The threshold per rectenna (expanded from scalar if needed).
    meets : np.ndarray of bool, shape (N_rect,)
        True if coverage[r] >= threshold[r] - atol.
    all_ok : bool
        True if all rectennas satisfy the constraint.
    """

    # convert data to numpy
    if isinstance(data, torch.Tensor):
        vals_full = data.cpu().numpy()
    else:
        vals_full = np.asarray(data, dtype=float)

    # Values for ratios only: shape (N_theta, N_raan, N_rect)
    vals = vals_full[..., 1]  # take column 1

    N_theta, N_raan, N_rect = vals.shape

    # handle threshold
    if np.isscalar(val_threshold):
        threshold = np.full(N_rect, float(val_threshold))
    else:
        threshold = np.asarray(val_threshold, dtype=float)
        if threshold.shape != (N_rect,):
            raise ValueError(
                f"val_threshold must be scalar or shape (N_rect,), got {threshold.shape}"
            )

    # accumulate coverage per rectenna
    coverage = np.zeros(N_rect, dtype=float)
    for (theta_idx, raan_idx) in selected_indices:
        if not (0 <= theta_idx < N_theta and 0 <= raan_idx < N_raan):
            raise IndexError(
                f"Selected index (theta_idx={theta_idx}, raan_idx={raan_idx}) "
                "is out of bounds for data of shape "
                f"(N_theta={N_theta}, N_raan={N_raan}, N_rect={N_rect})"
            )
        coverage += vals[theta_idx, raan_idx, :]

    meets = coverage >= (threshold - atol)
    all_ok = bool(np.all(meets))

    if print_details:
        print("\n=== Verification of MILP Solution ===")
        print(f"Number of selected (theta, raan) points: {len(selected_indices)}")
        print(f"All rectennas satisfy threshold? {all_ok}")
        print("\nPer-rectenna coverage:")
        for r in range(N_rect):
            status = "OK" if meets[r] else "FAIL"
            print(
                f"Rectenna {r:02d}: coverage = {coverage[r]:.6f}, "
                f"threshold = {threshold[r]:.6f} -> {status}"
            )

        min_margin = np.min(coverage - threshold)
        print(f"\nMinimum (coverage - threshold) across rectennas: {min_margin:.6f}")

    return coverage, threshold, meets, all_ok


Using device: cuda


In [3]:
# ========= User parameters =========
file_path = '/media/raj/New_Volume_F/SEM3/space_robotics/project_2/rectanna_placement.csv'

# Load rectenna placement (assume first two columns are [lat_deg, lon_deg])
data_rectanna_placement_degrees = pd.read_csv(file_path)
lat_long_array_deg = data_rectanna_placement_degrees.iloc[:, :2].values  # (N_rect, 2)

# Satellite grid definition
theta_list_deg = np.linspace(0.0, 90.0, 100)   # N_theta = 100
raan_list_deg  = np.linspace(0.0, 180.0, 100)  # N_raan  = 100
total_time_seconds = 24 * 3600 * 5             # 5 days

theta_threshold_deg = 3.0
chunk_size = 3600
orbit_height_m = 500e3

# ========= 1) Compute the satellite grid =========
result_matrix = compute_satellite_grid_torch(
    theta_list_deg=theta_list_deg,
    raan_list_deg=raan_list_deg,
    total_time_seconds=total_time_seconds,
    lat_long_array_deg=lat_long_array_deg,
    theta_threshold_deg=theta_threshold_deg,
    chunk_size=chunk_size,
    orbit_height_m=orbit_height_m,
)

print("Result matrix shape:", result_matrix.shape)  # (N_theta, N_raan, N_rect, 2)


# ========= 2) Solve MILP (minimum selection) =========
# You can tune this threshold
val_threshold = 0.85  # required summed ratio per rectenna

selected_indices, status = solve_minimum_selection(
    data=result_matrix,
    val_threshold=val_threshold,
    verbose=True,      # set False to suppress CBC logs
    gap_rel=0.01,      # 1% relative gap
    time_limit=1000,   # seconds (or None for unlimited)
)

print("\nMILP solver status:", status)
print("Number of selected (theta, raan) grid points:", len(selected_indices))
print("First few selected indices:", selected_indices[:10])


# ========= 3) Verify MILP solution =========
coverage, threshold_vec, meets, all_ok = verify_solution(
    data=result_matrix,
    selected_indices=selected_indices,
    val_threshold=val_threshold,
    atol=1e-6,
    print_details=True,
)

if all_ok:
    print("\n✅ Verification passed: all rectennas meet or exceed the required threshold.")
else:
    print("\n❌ Verification failed: at least one rectenna does NOT meet the required threshold.")


Theta-RAAN grid:   0%|                                                                                   | 0/10000 [00:00<?, ?it/s]/tmp/ipykernel_8337/1017280451.py:71: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:63.)
  f = torch.cross(K, u)
Theta-RAAN grid: 100%|███████████████████████████████████| 10000/10000 [16:04<00:00, 10.37it/s, elapsed=0h 16m 04s, eta=0h 00m 00s]


Total elapsed: 0h 16m 04s
Result matrix shape: torch.Size([100, 100, 50, 2])
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/raj/.local/lib/python3.10/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/183efe78af7e430888b750ee926b24f3-pulp.mps -sec 1000 -ratio 0.01 -timeMode elapsed -branch -printingOptions all -solution /tmp/183efe78af7e430888b750ee926b24f3-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 55 COLUMNS
At line 327856 RHS
At line 327907 BOUNDS
At line 337908 ENDATA
Problem MODEL has 50 rows, 10000 columns and 297800 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 1000
ratioGap was changed from 0 to 0.01
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1180.3 - 1.43 seconds
Cgl0004I processed model has 50 rows, 9838 columns (9838 integer (9786 of which binary)) and 297675 elements
Cutoff increment increased from 1e-05 to 0.9